In [1]:
pip install qiskit qiskit-aer

  Using cached qiskit-2.6.2-cp310-abi3-win_amd64.whl.metadata (14 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --------- ------------------------------ 2.1/9.3 MB 11.8 MB/s eta 0:00:01
   ------------- -------------------------- 3.1/9.3 MB 8.0 MB/s eta 0:00:01
   --------------------- ------------------ 5.0/9.3 MB 8.2 MB/s eta 0:00:01
   ------------------------------------ --- 8.4/9.3 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 9.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------------------- -------------------- 4.7/9.9 MB 20.3 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 20.8 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 15.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   --------------- ------------------------ 4.7/12.6 MB 25.9 MB/s eta 0:00:01
   ----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.5.5 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.5.5 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.18.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.5 which is incompatible.


In [2]:

from collections import deque
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Initial and goal states
# 0 represents the blank space
initial = (2, 3, 1, 0)
goal = (1, 2, 3, 0)

# Possible movements of the blank space
moves = {
    0: [1, 2],
    1: [0, 3],
    2: [0, 3],
    3: [1, 2]
}

# Display the puzzle
def display(state):
    for i in range(0, 4, 2):
        print(["_" if x == 0 else x for x in state[i:i+2]])

# Find solution using production rules
def solve_puzzle(start, goal):
    queue = deque([start])
    parent = {start: None}
    path = {}

    while queue:
        state = queue.popleft()

        if state == goal:
            break

        blank = state.index(0)

        for next_pos in moves[blank]:
            new_state = list(state)
            new_state[blank], new_state[next_pos] = (
                new_state[next_pos], new_state[blank]
            )
            new_state = tuple(new_state)

            if new_state not in parent:
                parent[new_state] = state
                path[new_state] = (blank, next_pos)
                queue.append(new_state)

    if goal not in parent:
        return None

    solution = []
    state = goal

    while parent[state] is not None:
        solution.append(path[state])
        state = parent[state]

    return solution[::-1]


# Find the solution
solution = solve_puzzle(initial, goal)

print("Initial State:")
display(initial)

print("\nNumber of Moves:", len(solution))

# Create an 8-qubit quantum circuit
qc = QuantumCircuit(8, 8)

# Encode the initial board state
# 00 = Blank, 01 = 1, 10 = 2, 11 = 3
for cell, tile in enumerate(initial):
    binary = format(tile, "02b")

    for j, bit in enumerate(binary):
        if bit == "1":
            qc.x(2 * cell + j)

qc.barrier()

# Apply reversible SWAP gates for each production rule
for blank, next_pos in solution:
    qc.swap(2 * blank, 2 * next_pos)
    qc.swap(2 * blank + 1, 2 * next_pos + 1)
    qc.barrier()

# Measure all qubits
qc.measure(range(8), range(8))

print("\nQuantum Circuit:")
print(qc.draw())

# Simulate the quantum circuit
simulator = AerSimulator()
compiled_circuit = transpile(qc, simulator)

result = simulator.run(
    compiled_circuit, shots=1024
).result()

counts = result.get_counts()

# Obtain the measured solution
measured = max(counts, key=counts.get)[::-1]

final_state = [
    int(measured[i:i+2], 2)
    for i in range(0, 8, 2)
]

print("\nMeasurement Results:", counts)

print("\nFinal Solution:")
display(final_state)


Initial State:
[2, 3]
[1, '_']

Number of Moves: 4

Quantum Circuit:
     ┌───┐ ░        ░        ░        ░        ░ ┌─┐                     
q_0: ┤ X ├─░────────░──X─────░──X─────░────────░─┤M├─────────────────────
     └───┘ ░        ░  │     ░  │     ░        ░ └╥┘┌─┐                  
q_1: ──────░────────░──┼──X──░──┼──X──░────────░──╫─┤M├──────────────────
     ┌───┐ ░        ░  │  │  ░  │  │  ░        ░  ║ └╥┘┌─┐               
q_2: ┤ X ├─░──X─────░──X──┼──░──┼──┼──░────────░──╫──╫─┤M├───────────────
     ├───┤ ░  │     ░     │  ░  │  │  ░        ░  ║  ║ └╥┘┌─┐            
q_3: ┤ X ├─░──┼──X──░─────X──░──┼──┼──░────────░──╫──╫──╫─┤M├────────────
     └───┘ ░  │  │  ░        ░  │  │  ░        ░  ║  ║  ║ └╥┘┌─┐         
q_4: ──────░──┼──┼──░────────░──X──┼──░──X─────░──╫──╫──╫──╫─┤M├─────────
     ┌───┐ ░  │  │  ░        ░     │  ░  │     ░  ║  ║  ║  ║ └╥┘┌─┐      
q_5: ┤ X ├─░──┼──┼──░────────░─────X──░──┼──X──░──╫──╫──╫──╫──╫─┤M├──────
     └───┘ ░  │  │  ░        ░        ░  │ 